# Stable Traditional ML Pipeline

This notebook is the safest version for your dataset and laptop.

It fixes the earlier problems by:
- not using `matplotlib`
- not loading all parquet image data into RAM at once
- testing image columns automatically
- handling nested and ragged image arrays more safely
- running `GLCM + LBP + Naive Bayes + SVM`

In [1]:
!pip install pandas pyarrow pillow numpy scikit-learn scikit-image tqdm

In [2]:
from __future__ import annotations

import io
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from tqdm import tqdm

In [3]:
PARQUET_PATTERN = '*.parquet'

# Start low for reliability. Increase later if it runs fine.
MAX_SAMPLES = 20

# Preferred image column. The notebook will auto-fallback if this one fails.
PREFERRED_IMAGE_COL = 'Image_Pre'

# Keep None to build labels from Lesion_Left + Lesion_Right.
# If you have a true class column later, put its name here.
LABEL_COL = None

IMAGE_SIZE = (96, 96)
LBP_RADIUS = 2
LBP_POINTS = 8 * LBP_RADIUS
LBP_METHOD = 'uniform'
GLCM_DISTANCES = [1, 2]
GLCM_ANGLES = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]
RANDOM_STATE = 42
TEST_SIZE = 0.2

SEARCH_ROOTS = [
    Path.cwd(),
    Path.home(),
    Path('/content'),
]

In [4]:
def auto_find_data_dir(search_roots: list[Path], pattern: str = '*.parquet') -> Path:
    for root in search_roots:
        try:
            if root.exists() and list(root.glob(pattern)):
                return root
        except Exception:
            pass

    for root in search_roots:
        try:
            if not root.exists():
                continue
            for path in root.rglob(pattern):
                return path.parent
        except Exception:
            pass

    raise FileNotFoundError('No parquet files found. Put this notebook in the same folder as the parquet files, or set DATA_DIR manually.')


def find_parquet_files(data_dir: Path, pattern: str = '*.parquet') -> list[Path]:
    files = sorted(data_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No parquet files found in {data_dir}')
    return files


def get_schema_columns(parquet_file: Path) -> list[str]:
    return pq.ParquetFile(parquet_file).schema.names


def candidate_image_columns(columns: list[str], preferred: str = 'Image_Pre') -> list[str]:
    priority = [
        preferred,
        'Image_Pre', 'Image_Post_1', 'Image_Post_2', 'Image_Post_3',
        'Image_Post_4', 'Image_Post_5', 'Image_Post_6', 'Image_Post_7',
        'Image_Sub_1', 'Image_T2',
        'image', 'img', 'images', 'pixel_values', 'input_image',
        'image_path', 'filepath', 'file_path', 'path'
    ]
    result = []
    seen = set()
    for col in priority:
        if col in columns and col not in seen:
            result.append(col)
            seen.add(col)
    return result


def normalize_value(value: Any) -> str:
    if value is None:
        return 'missing'

    if isinstance(value, np.ndarray):
        if value.size == 0:
            return 'missing'
        if value.size == 1:
            return str(value.item())
        return str(value.tolist())

    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            return 'missing'
        if len(value) == 1:
            return str(value[0])
        return str(list(value))

    try:
        if pd.isna(value):
            return 'missing'
    except Exception:
        pass

    if isinstance(value, (np.integer, int)):
        return str(int(value))

    if isinstance(value, (np.floating, float)):
        if float(value).is_integer():
            return str(int(value))
        return str(float(value))

    text = str(value).strip()
    return text if text else 'missing'


def build_label_from_row(row: dict[str, Any], label_col: str | None = None) -> str:
    if label_col is not None:
        return normalize_value(row.get(label_col))

    left = normalize_value(row.get('Lesion_Left'))
    right = normalize_value(row.get('Lesion_Right'))
    return f'Left_{left}__Right_{right}'


def iter_rows_from_parquet(parquet_file: Path, columns: list[str], batch_size: int = 1):
    pf = pq.ParquetFile(parquet_file)
    for batch in pf.iter_batches(columns=columns, batch_size=batch_size):
        data = batch.to_pydict()
        length = len(next(iter(data.values())))
        for i in range(length):
            yield {col: data[col][i] for col in data}


def try_numeric_array_direct(value: Any):
    try:
        arr = np.asarray(value)
        if arr.dtype != object:
            return arr
    except Exception:
        pass
    return None


def coerce_numeric_array(value: Any) -> np.ndarray:
    direct = try_numeric_array_direct(value)
    if direct is not None:
        return direct

    if isinstance(value, np.ndarray):
        if value.ndim == 0:
            return coerce_numeric_array(value.item())
        for item in value.flat:
            try:
                return coerce_numeric_array(item)
            except Exception:
                pass
        raise TypeError('Could not convert object ndarray into a numeric image array')

    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            raise TypeError('Empty list/tuple cannot be converted to image')
        nested = []
        for item in value:
            try:
                nested.append(coerce_numeric_array(item))
            except Exception:
                pass
        if not nested:
            raise TypeError('No numeric child arrays found in nested sequence')
        return np.asarray(max(nested, key=lambda x: np.asarray(x).size))

    if isinstance(value, dict):
        for key in ['array', 'data', 'pixel_array', 'values']:
            if key in value and value[key] is not None:
                return coerce_numeric_array(value[key])
        raise TypeError(f'Unsupported dict image payload with keys: {list(value.keys())}')

    if isinstance(value, (np.integer, int, np.floating, float)):
        return np.asarray([[value]], dtype=np.float32)

    raise TypeError(f'Unsupported value for numeric array conversion: {type(value)}')


def normalize_to_uint8(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    arr = np.nan_to_num(arr.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    if arr.size == 0:
        raise ValueError('Empty array after conversion')
    arr_min = float(arr.min())
    arr_max = float(arr.max())
    if arr_max > arr_min:
        arr = (arr - arr_min) / (arr_max - arr_min)
    else:
        arr = np.zeros_like(arr, dtype=np.float32)
    return (arr * 255).clip(0, 255).astype(np.uint8)


def array_to_pil_image(arr: np.ndarray) -> Image.Image:
    arr = np.asarray(arr)
    arr = np.squeeze(arr)

    while arr.ndim > 3:
        arr = arr[arr.shape[0] // 2]

    if arr.ndim == 3:
        if arr.shape[-1] in [1, 3, 4]:
            arr = normalize_to_uint8(arr)
            if arr.shape[-1] == 1:
                return Image.fromarray(arr[:, :, 0]).convert('RGB')
            return Image.fromarray(arr[:, :, :3]).convert('RGB')

        if arr.shape[0] in [1, 3, 4] and arr.shape[-1] not in [1, 3, 4]:
            arr = np.moveaxis(arr, 0, -1)
            arr = normalize_to_uint8(arr)
            if arr.shape[-1] == 1:
                return Image.fromarray(arr[:, :, 0]).convert('RGB')
            return Image.fromarray(arr[:, :, :3]).convert('RGB')

        arr = arr[arr.shape[0] // 2]

    if arr.ndim == 2:
        arr = normalize_to_uint8(arr)
        return Image.fromarray(arr).convert('RGB')

    if arr.ndim == 1:
        side = int(np.sqrt(arr.size))
        if side * side == arr.size:
            arr = arr.reshape(side, side)
            arr = normalize_to_uint8(arr)
            return Image.fromarray(arr).convert('RGB')

    raise TypeError(f'Could not convert array with shape {arr.shape} into an image')


def decode_image(value: Any) -> Image.Image:
    if isinstance(value, Image.Image):
        return value.convert('RGB')

    if isinstance(value, (bytes, bytearray)):
        return Image.open(io.BytesIO(value)).convert('RGB')

    if isinstance(value, str):
        return Image.open(value).convert('RGB')

    if isinstance(value, dict):
        if 'bytes' in value and value['bytes'] is not None:
            return Image.open(io.BytesIO(value['bytes'])).convert('RGB')
        if 'path' in value and value['path'] is not None:
            return Image.open(value['path']).convert('RGB')

    arr = coerce_numeric_array(value)
    return array_to_pil_image(arr)


def preprocess_image(value: Any, image_size: tuple[int, int] = IMAGE_SIZE) -> np.ndarray:
    image = decode_image(value)
    image = image.convert('L').resize(image_size)
    return np.array(image, dtype=np.uint8)


def find_working_image_column(parquet_files: list[Path], candidates: list[str], sample_checks: int = 5) -> str:
    last_errors = []
    for candidate in candidates:
        success = 0
        for file in parquet_files:
            for row in iter_rows_from_parquet(file, [candidate], batch_size=1):
                try:
                    _ = preprocess_image(row[candidate], IMAGE_SIZE)
                    success += 1
                    if success >= sample_checks:
                        return candidate
                except Exception as exc:
                    last_errors.append(f'{candidate}: {exc}')
                    if len(last_errors) > 10:
                        last_errors = last_errors[-10:]
                if success + len(last_errors) >= sample_checks:
                    break
            if success + len(last_errors) >= sample_checks:
                break

    error_text = '\n'.join(last_errors[-5:]) if last_errors else 'No candidate image columns succeeded.'
    raise ValueError(f'Could not find a decodable image column. Recent errors:\n{error_text}')


def extract_glcm_features(gray_image: np.ndarray) -> np.ndarray:
    glcm = graycomatrix(
        gray_image,
        distances=GLCM_DISTANCES,
        angles=GLCM_ANGLES,
        levels=256,
        symmetric=True,
        normed=True,
    )
    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    features = []
    for prop in props:
        features.extend(graycoprops(glcm, prop).flatten())
    return np.array(features, dtype=np.float32)


def extract_lbp_features(gray_image: np.ndarray) -> np.ndarray:
    lbp = local_binary_pattern(gray_image, LBP_POINTS, LBP_RADIUS, method=LBP_METHOD)
    n_bins = LBP_POINTS + 2
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1), range=(0, n_bins))
    hist = hist.astype(np.float32)
    hist /= hist.sum() + 1e-8
    return hist


def extract_combined_features(gray_image: np.ndarray) -> np.ndarray:
    return np.concatenate([extract_glcm_features(gray_image), extract_lbp_features(gray_image)]).astype(np.float32)


def can_stratify(y: np.ndarray) -> bool:
    values, counts = np.unique(y, return_counts=True)
    return len(values) > 1 and counts.min() >= 2

In [ ]:
DATA_DIR = auto_find_data_dir(SEARCH_ROOTS, PARQUET_PATTERN)
print('Using DATA_DIR:', DATA_DIR)

parquet_files = find_parquet_files(DATA_DIR, PARQUET_PATTERN)
print(f'Found {len(parquet_files)} parquet file(s)')

all_columns = get_schema_columns(parquet_files[0])
print('Columns:', all_columns)

image_candidates = candidate_image_columns(all_columns, PREFERRED_IMAGE_COL)
print('Image column candidates:', image_candidates)

if not image_candidates:

    raise ValueError('No known image columns were found in this dataset.')

image_col = find_working_image_column(parquet_files, image_candidates)
print('Selected working image column:', image_col)

cols_to_read = [image_col]
if LABEL_COL is not None:
    cols_to_read.append(LABEL_COL)
else:
    cols_to_read.extend(['Lesion_Left', 'Lesion_Right'])

print('Reading columns:', cols_to_read)

Using DATA_DIR: c:\Users\Ayush\Desktop\dmwatt\dataset
Found 14 parquet file(s)
Columns: ['UID', 'Fold', 'Split', 'PatientID', 'Institution', 'Age', 'Lesion_Left', 'Lesion_Right', '__index_level_0__', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'element']
Image column candidates: []


ValueError: Could not find a decodable image column. Recent errors:
No candidate image columns succeeded.

In [6]:
X_features = []
y_values = []
processed = 0
skipped = 0
shown_errors = 0

for file in tqdm(parquet_files, desc='Parquet files'):
    if MAX_SAMPLES is not None and processed >= MAX_SAMPLES:
        break

    for row in iter_rows_from_parquet(file, cols_to_read, batch_size=1):
        if MAX_SAMPLES is not None and processed >= MAX_SAMPLES:
            break

        try:
            gray_image = preprocess_image(row[image_col], IMAGE_SIZE)
            features = extract_combined_features(gray_image)
            label = build_label_from_row(row, LABEL_COL)

            X_features.append(features)
            y_values.append(label)
            processed += 1
        except Exception as exc:
            skipped += 1
            if shown_errors < 5:
                print(f'Skipping one row in {file.name} due to error: {exc}')
                shown_errors += 1

if not X_features:
    raise ValueError('No samples were processed. Reduce IMAGE_SIZE, lower MAX_SAMPLES, or change PREFERRED_IMAGE_COL.')

X = np.vstack(X_features)
y_values = np.array(y_values)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_values)

print('Processed samples:', processed)
print('Skipped samples:', skipped)
print('Feature matrix shape:', X.shape)
print('Classes:', list(label_encoder.classes_))

Parquet files:   0%|          | 0/14 [00:00<?, ?it/s]


NameError: name 'cols_to_read' is not defined

In [7]:
if len(X) < 4:
    raise ValueError('Too few processed samples. Increase MAX_SAMPLES or switch image column.')

if len(np.unique(y)) < 2:
    raise ValueError('Only one class found in processed samples. Increase MAX_SAMPLES or use a different label column.')

stratify_arg = y if can_stratify(y) else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_arg,
)

print('Train shape:', X_train.shape, y_train.shape)
print('Test shape:', X_test.shape, y_test.shape)
print('Stratified split used:', stratify_arg is not None)

NameError: name 'X' is not defined

In [ ]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

all_label_ids = np.arange(len(label_encoder.classes_))

print('Naive Bayes Accuracy:', accuracy_score(y_test, nb_pred))
print('\nNaive Bayes Classification Report:\n')
print(classification_report(y_test, nb_pred, labels=all_label_ids, target_names=label_encoder.classes_, zero_division=0))
print('Naive Bayes Confusion Matrix:\n', confusion_matrix(y_test, nb_pred, labels=all_label_ids))

In [ ]:
svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=10, gamma='scale')),
])

svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)

print('SVM Accuracy:', accuracy_score(y_test, svm_pred))
print('\nSVM Classification Report:\n')
print(classification_report(y_test, svm_pred, labels=all_label_ids, target_names=label_encoder.classes_, zero_division=0))
print('SVM Confusion Matrix:\n', confusion_matrix(y_test, svm_pred, labels=all_label_ids))

In [ ]:
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM'],
    'Accuracy': [accuracy_score(y_test, nb_pred), accuracy_score(y_test, svm_pred)]
}).sort_values('Accuracy', ascending=False)

results

## If you still need lighter settings

Change only the config cell to:

```python
MAX_SAMPLES = 10
IMAGE_SIZE = (64, 64)
PREFERRED_IMAGE_COL = 'Image_Post_1'
```

This notebook will still auto-test the image columns and pick the first one that decodes correctly.